# Анализ данных

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Инициализация Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

# Создаем Spark сессию
spark = SparkSession.builder \
    .appName("StudentActivityAnalysis") \
    .getOrCreate()

# Загружаем данные
df = spark.read.csv('/content/drive/MyDrive/data_sets/aggrigation_logs_per_week.csv',
                    header=True, inferSchema=True)

# Преобразуем строки с запятыми в числа
columns_to_fix = ['s_all_avg', 's_course_viewed_avg', 's_q_attempt_viewed_avg',
                  's_a_course_module_viewed_avg', 's_a_submission_status_viewed_avg']

for col_name in columns_to_fix:
    if col_name in df.columns:
        df = df.withColumn(col_name, regexp_replace(col(col_name), ',', '.').cast('float'))

In [3]:
unique_students = df.select("userid").distinct().count()
print(f"Всего уникальных студентов: {unique_students}")

total_records = df.count()
print(f"Всего записей: {total_records}")

Всего уникальных студентов: 6316
Всего записей: 414528


In [4]:

print("ЗАДАНИЕ 1: Анализ активности студентов по неделям")

weekly_activity = df.groupBy("num_week") \
    .agg(
        sum("s_all").alias("total_events"),
        sum("s_course_viewed").alias("total_course_views"),
        count("userid").alias("record_count")
    ) \
    .orderBy("num_week")

print("Активность по неделям:")
weekly_activity.show()

# Анализ
print("\nАНАЛИЗ:")
max_week = weekly_activity.orderBy(desc("total_events")).first()
print(f"1. Самая активная неделя: {max_week['num_week']}")
print(f"2. Всего событий на этой неделе: {max_week['total_events']}")
print("3. Пик активности на 6-й неделе, затем спад")

ЗАДАНИЕ 1: Анализ активности студентов по неделям
Активность по неделям:
+--------+------------+------------------+------------+
|num_week|total_events|total_course_views|record_count|
+--------+------------+------------------+------------+
|       6|      238295|             75540|       17272|
|       7|      166090|             47988|       17272|
|       8|      138669|             37077|       17272|
|       9|      162241|             40142|       17272|
|      10|      141778|             33449|       17272|
|      11|      173100|             39493|       17272|
|      12|      162041|             37901|       17272|
|      13|      172962|             38890|       17272|
|      14|      170305|             37556|       17272|
|      15|      178829|             39322|       17272|
|      16|      177650|             38385|       17272|
|      17|      181629|             37328|       17272|
|      18|      167044|             34461|       17272|
|      19|      191952|        

In [6]:
from pyspark.sql.functions import countDistinct, avg, desc

print("ЗАДАНИЕ 2: Самые популярные курсы")

top_courses = df.groupBy("courseid") \
    .agg(
        avg("s_course_viewed_avg").alias("avg_course_views"),
        countDistinct("userid").alias("unique_students")
    ) \
    .orderBy(desc("avg_course_views")) \
    .limit(5)

print("Топ-5 курсов по средним просмотрам:")
top_courses.show()

ЗАДАНИЕ 2: Самые популярные курсы
Топ-5 курсов по средним просмотрам:
+--------+------------------+---------------+
|courseid|  avg_course_views|unique_students|
+--------+------------------+---------------+
|   76419|31.477879250800708|             41|
|   78733|24.643542336041314|             12|
|   78705|21.971443291458613|             20|
|   82552|17.460925520708162|             24|
|   73823| 15.80914722672767|             24|
+--------+------------------+---------------+



In [7]:

print("ЗАДАНИЕ 3: Связь просмотров и количества студентов")

course_stats = df.groupBy("courseid") \
    .agg(
        avg("s_course_viewed").alias("avg_views"),
        countDistinct("userid").alias("student_count")
    )

print("Статистика по курсам:")
course_stats.show()

# Корреляция
correlation = course_stats.select(
    corr("avg_views", "student_count").alias("correlation")
)

print("Корреляция между средними просмотрами и количеством студентов:")
correlation.show()

ЗАДАНИЕ 3: Связь просмотров и количества студентов
Статистика по курсам:
+--------+-------------------+-------------+
|courseid|          avg_views|student_count|
+--------+-------------------+-------------+
|   71995|  2.071759259259259|           18|
|   72820|0.20052083333333334|           16|
|   75705| 1.8333333333333333|            4|
|   78120| 0.9861111111111112|            3|
|   82672| 1.7645833333333334|           20|
|   71709| 0.7481884057971014|           46|
|   71987|  1.368421052631579|           19|
|   78272| 1.0112179487179487|           26|
|   89379|0.42916666666666664|           10|
|   72232| 2.3645833333333335|           12|
|   71732| 1.0506756756756757|           37|
|   75969|             0.5875|           10|
|   89044| 0.6145833333333334|            8|
|   84275| 0.6130952380952381|           21|
|   79130|   6.78030303030303|           11|
|   89165| 0.9886363636363636|           11|
|   73341| 1.0705128205128205|           13|
|   76293|  3.2559523809523

In [8]:

print("ЗАДАНИЕ 4: Сравнение активности на бюджете и контракте")

# Создаем категорию
df_with_type = df.withColumn(
    "study_type",
    when(col("Name_OsnO") == 1, "бюджет")
    .when(col("Name_OsnO") == 2, "контракт")
    .otherwise("другое")
)

budget_analysis = df_with_type.groupBy("study_type") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        countDistinct("userid").alias("student_count")
    )

print("\nАктивность по типам финансирования:")
budget_analysis.show()

# Разница (если есть обе группы)
budget_data = budget_analysis.filter(col("study_type") == "бюджет")
contract_data = budget_analysis.filter(col("study_type") == "контракт")

if budget_data.count() > 0 and contract_data.count() > 0:
    budget_avg = budget_data.first()["avg_activity"]
    contract_avg = contract_data.first()["avg_activity"]
    diff = budget_avg - contract_avg
    print(f"\nРазница в активности: {diff:.2f}")

ЗАДАНИЕ 4: Сравнение активности на бюджете и контракте

Активность по типам финансирования:
+----------+------------------+-------------+
|study_type|      avg_activity|student_count|
+----------+------------------+-------------+
|  контракт| 8.275873273760482|         2006|
|    бюджет|12.023394173545041|         4310|
+----------+------------------+-------------+


Разница в активности: 3.75


In [9]:

print("ЗАДАНИЕ 5: Влияние формы обучения")

form_analysis = df.groupBy("Name_FormOPril") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        stddev("s_all_avg").alias("std_activity"),
        countDistinct("userid").alias("student_count")
    ) \
    .orderBy("Name_FormOPril")

print("Активность по формам обучения:")
form_analysis.show()

ЗАДАНИЕ 5: Влияние формы обучения
Активность по формам обучения:
+--------------+------------------+------------------+-------------+
|Name_FormOPril|      avg_activity|      std_activity|student_count|
+--------------+------------------+------------------+-------------+
|             1|14.350096475349721|24.905277377178535|         3917|
|             2|3.8553863223734757|  9.76752103698596|         2381|
|             3|3.4267937415538148| 14.13102038084555|           18|
+--------------+------------------+------------------+-------------+



In [10]:

print("ЗАДАНИЕ 6: Активность по семестрам")

semester_activity = df.groupBy("Num_Sem") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        sum("s_all").alias("total_events"),
        countDistinct("userid").alias("student_count")
    ) \
    .orderBy(desc("avg_activity"))

print("Активность по семестрам:")
semester_activity.show()

best_semester = semester_activity.first()
print(f"\nСамый активный семестр: {best_semester['Num_Sem']}")
print(f"Средняя активность: {best_semester['avg_activity']:.2f}")

ЗАДАНИЕ 6: Активность по семестрам
Активность по семестрам:
+-------+------------------+------------+-------------+
|Num_Sem|      avg_activity|total_events|student_count|
+-------+------------------+------------+-------------+
|      2|14.681787366892909|     2070141|         2064|
|      4|  9.91017145405209|     1061751|         1621|
|      6| 9.241324991333808|      934114|         1597|
|      8| 8.343872770473682|      533810|          812|
|     10| 3.929059153925176|       46513|          162|
|     12|0.5502061953957098|        3116|           60|
+-------+------------------+------------+-------------+


Самый активный семестр: 2
Средняя активность: 14.68


In [11]:

print("ЗАДАНИЕ 7: Топ-3 кафедры по активности")


department_activity = df.groupBy("Depart") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        countDistinct("userid").alias("student_count"),
        countDistinct("courseid").alias("course_count")
    ) \
    .orderBy(desc("avg_activity")) \
    .limit(3)

print("Топ-3 кафедры по активности:")
department_activity.show()

ЗАДАНИЕ 7: Топ-3 кафедры по активности
Топ-3 кафедры по активности:
+------+------------------+-------------+------------+
|Depart|      avg_activity|student_count|course_count|
+------+------------------+-------------+------------+
|     4|30.511041890529494|          235|          35|
|    24|24.805201334660403|          125|          23|
|    12| 22.13958671334555|          127|          16|
+------+------------------+-------------+------------+



In [13]:

print("ЗАДАНИЕ 8: Успеваемость в зависимости от активности")

# Создаем категории активности
df_with_categories = df.withColumn(
    "activity_category",
    when(col("s_all_avg") < 2, "очень низкая")
    .when((col("s_all_avg") >= 2) & (col("s_all_avg") < 5), "низкая")
    .when((col("s_all_avg") >= 5) & (col("s_all_avg") < 10), "средняя")
    .otherwise("высокая")
)

performance_analysis = df_with_categories.groupBy("activity_category") \
    .agg(
        avg("NameR_Level").alias("avg_grade"),
        avg("s_all_avg").alias("avg_activity"),
        countDistinct("userid").alias("student_count")
    ) \
    .orderBy("avg_grade")

print("Успеваемость по уровням активности:")
performance_analysis.show()

# Корреляция
corr_result = df.select(
    corr("s_all_avg", "NameR_Level").alias("correlation_activity_grade")
)
print("Корреляция между активностью и оценками:")
corr_result.show()

ЗАДАНИЕ 8: Успеваемость в зависимости от активности
Успеваемость по уровням активности:
+-----------------+------------------+-------------------+-------------+
|activity_category|         avg_grade|       avg_activity|student_count|
+-----------------+------------------+-------------------+-------------+
|     очень низкая|  3.72422759601707|0.45868968032584506|         5628|
|           низкая|4.0448614006640415|  3.274096185262116|         4901|
|          средняя| 4.223542419881357|  7.156617454891814|         4625|
|          высокая| 4.372719894057841|  32.85979273419362|         4206|
+-----------------+------------------+-------------------+-------------+

Корреляция между активностью и оценками:
+--------------------------+
|correlation_activity_grade|
+--------------------------+
|        0.1754915346726081|
+--------------------------+



In [15]:
print("ЗАДАНИЕ 9: Студенты с аномально низкой активностью")

window_spec = Window.partitionBy("Kurs")

df_with_avg = df.withColumn(
    "kurs_avg_activity",
    avg("s_all_avg").over(window_spec)
)

low_activity_records = df_with_avg.filter(
    col("s_all_avg") < col("kurs_avg_activity")
)

unique_low_activity_students = low_activity_records.select("userid").distinct().count()
total_students = df.select("userid").distinct().count()

print(f"Всего студентов в базе: {total_students}")
print(f"Найдено УНИКАЛЬНЫХ студентов с низкой активностью: {unique_low_activity_students}")
print(f"Процент студентов с низкой активностью: {unique_low_activity_students / total_students * 100:.1f}%")

low_activity_examples = low_activity_records \
    .groupBy("userid") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        first("kurs_avg_activity").alias("course_avg"),
        avg("NameR_Level").alias("avg_grade"),
        countDistinct("courseid").alias("courses_count")
    ) \
    .orderBy("avg_activity") \
    .limit(10)

print("\nПримеры студентов с низкой активностью:")
low_activity_examples.show()

low_activity_detail = low_activity_examples \
    .withColumn(
        "diff_percent",
        ((col("course_avg") - col("avg_activity")) / col("course_avg") * 100)
    ) \
    .orderBy(desc("diff_percent"))

print("\nТоп студентов с самой низкой активностью:")
low_activity_detail.select(
    "userid",
    "avg_activity",
    "course_avg",
    "avg_grade",
    "courses_count",
    round("diff_percent", 2).alias("diff_percent")
).show()

ЗАДАНИЕ 9: Студенты с аномально низкой активностью
Всего студентов в базе: 6316
Найдено УНИКАЛЬНЫХ студентов с низкой активностью: 6228
Процент студентов с низкой активностью: 98.6%

Примеры студентов с низкой активностью:
+------+------------+------------------+-----------------+-------------+
|userid|avg_activity|        course_avg|        avg_grade|courses_count|
+------+------------+------------------+-----------------+-------------+
| 28976|         0.0|  9.91017145405209|              5.0|            1|
| 36074|         0.0|  9.91017145405209|              2.0|            1|
| 23103|         0.0| 8.343872770473682|4.333333333333333|            3|
| 34299|         0.0|14.681787366892909|              5.0|            1|
| 27428|         0.0| 9.241324991333808|              4.0|            1|
| 33426|         0.0|14.681787366892909|              4.0|            1|
| 26810|         0.0| 9.241324991333808|              3.0|            1|
| 27099|         0.0| 9.241324991333808|       

In [17]:

print("ЗАДАНИЕ 10: Кластеризация студентов по активности")

# 1. Агрегируем данные по студентам
student_features = df.groupBy("userid") \
    .agg(
        avg("s_all_avg").alias("avg_activity"),
        avg("s_course_viewed_avg").alias("avg_course_views"),
        avg("s_q_attempt_viewed_avg").alias("avg_quiz_attempts"),
        avg("NameR_Level").alias("avg_grade"),
        countDistinct("courseid").alias("courses_count")
    )

print("Агрегированные данные по студентам (первые 10):")
student_features.show(10)
print(f"Всего уникальных студентов: {student_features.count()}")

# 2. Подготавливаем фичи для кластеризации
feature_columns = ["avg_activity", "avg_course_views", "avg_quiz_attempts", "avg_grade"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
feature_data = assembler.transform(student_features)

# 3. Масштабируем данные
scaler = StandardScaler(inputCol="features", outputCol="scaled_features",
                       withStd=True, withMean=True)
scaler_model = scaler.fit(feature_data)
scaled_data = scaler_model.transform(feature_data)

# 4. Кластеризация K-Means
kmeans = KMeans(k=3, seed=42, featuresCol="scaled_features")
model = kmeans.fit(scaled_data)
clustered_data = model.transform(scaled_data)

print("\nРезультаты кластеризации (первые 10 студентов):")
clustered_data.select("userid", "avg_activity", "prediction").show(10)

# 5. Анализ кластеров
cluster_stats = clustered_data.groupBy("prediction") \
    .agg(
        avg("avg_activity").alias("avg_activity"),
        avg("avg_course_views").alias("avg_course_views"),
        avg("avg_quiz_attempts").alias("avg_quiz_attempts"),
        avg("avg_grade").alias("avg_grade"),
        count("userid").alias("student_count")
    ) \
    .orderBy("prediction")

print("\nХарактеристики кластеров:")
cluster_stats.show()

# 6. Интерпретация
print("\n📊 ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ:")

# Находим центры кластеров для интерпретации
cluster_centers = model.clusterCenters()
for i, center in enumerate(cluster_centers):
    print(f"\nКластер {i}:")
    print(f"  Количество студентов: {cluster_stats.filter(col('prediction') == i).first()['student_count']}")
    print(f"  Центр кластера: {center}")

    # Интерпретация по значению avg_activity
    if center[0] < -0.5:
        print(f"  Уровень активности: Низкий")
    elif center[0] < 0.5:
        print(f"  Уровень активности: Средний")
    else:
        print(f"  Уровень активности: Высокий")

# 7. Распределение студентов
distribution = clustered_data.groupBy("prediction").count().orderBy("prediction")
print("\nРаспределение студентов по кластерам:")
distribution.show()

# Сохраняем результат для дальнейшего анализа
clustered_data.write.mode("overwrite").parquet("/content/drive/MyDrive/data_sets/clustered_students.parquet")
print("\n✅ Результаты кластеризации сохранены в Parquet файл")

ЗАДАНИЕ 10: Кластеризация студентов по активности
Агрегированные данные по студентам (первые 10):
+------+-------------------+-------------------+--------------------+------------------+-------------+
|userid|       avg_activity|   avg_course_views|   avg_quiz_attempts|         avg_grade|courses_count|
+------+-------------------+-------------------+--------------------+------------------+-------------+
| 34234|  32.04439024130503|  8.932612551583183|   0.934706942902671| 4.333333333333333|            3|
| 20735|  9.144977763709095| 3.2395930575827756|  0.9729083379109701|               4.0|            3|
| 32445|0.16230000058809915|0.03769722249772814|0.054455555975437164|3.3333333333333335|            3|
| 28024|  2.133917704845468| 0.9756635394878685|0.059433333575725555|              3.25|            4|
| 33717| 10.751167726702988| 2.6199479224160314|  0.0690520831073324|               5.0|            4|
| 24347| 13.356364196538925| 3.5645099937915803|  2.0937691683570545|         

In [ ]:
print("\n" + "="*80)
print("СВОДКА РЕЗУЛЬТАТОВ ПО ВСЕМ ЗАДАНИЯМ")
print("="*80)

print("""
✅ ЗАДАНИЕ 1: Определена активность по неделям (пик на 6-й неделе)
✅ ЗАДАНИЕ 2: Найден топ-5 курсов по просмотрам
✅ ЗАДАНИЕ 3: Проанализирована связь просмотров и количества студентов
✅ ЗАДАНИЕ 4: Сравнена активность бюджета и контракта
✅ ЗАДАНИЕ 5: Исследовано влияние формы обучения
✅ ЗАДАНИЕ 6: Определен самый активный семестр
✅ ЗАДАНИЕ 7: Найдены топ-3 кафедры по активности
✅ ЗАДАНИЕ 8: Проанализирована связь активности и успеваемости
✅ ЗАДАНИЕ 9: Выявлены студенты с низкой активностью
✅ ЗАДАНИЕ 10: Выполнена кластеризация на 3 группы

ВСЕ ЗАДАНИЯ ВЫПОЛНЕНЫ НА PYSPARK!
Использованы: groupBy, agg, window functions, корреляции, K-Means кластеризация
""")

# Останавливаем Spark сессию
spark.stop()
print("Spark сессия остановлена")


СВОДКА РЕЗУЛЬТАТОВ ПО ВСЕМ ЗАДАНИЯМ

✅ ЗАДАНИЕ 1: Определена активность по неделям (пик на 6-й неделе)
✅ ЗАДАНИЕ 2: Найден топ-5 курсов по просмотрам
✅ ЗАДАНИЕ 3: Проанализирована связь просмотров и количества студентов
✅ ЗАДАНИЕ 4: Сравнена активность бюджета и контракта
✅ ЗАДАНИЕ 5: Исследовано влияние формы обучения
✅ ЗАДАНИЕ 6: Определен самый активный семестр
✅ ЗАДАНИЕ 7: Найдены топ-3 кафедры по активности
✅ ЗАДАНИЕ 8: Проанализирована связь активности и успеваемости
✅ ЗАДАНИЕ 9: Выявлены студенты с низкой активностью
✅ ЗАДАНИЕ 10: Выполнена кластеризация на 3 группы

ВСЕ ЗАДАНИЯ ВЫПОЛНЕНЫ НА PYSPARK!
Использованы: groupBy, agg, window functions, корреляции, K-Means кластеризация

Spark сессия остановлена
